## Load data and helper functions

In [ ]:
import pandas as pd 
import numpy as np 
import hashlib
import os
from datetime import datetime

# ==========================================
# Load data
# ==========================================

survey_names = open("survey_listN.txt", "r").read().strip().split('\n')
register = pd.read_excel("public_data_registerN.xlsx")
anon_data = pd.read_csv("anonymised_dataN.csv")

ref_date = datetime(2025, 7, 1)  # Reference date for age calculation (from report)

# ========================================== 
# DOB conversion function
# ==========================================
def dob_to_age_band(dob):
    age = ref_date.year - dob.year - ((dob.month, dob.day) > (ref_date.month, ref_date.day))
    if age < 30:
        return '18-29'
    elif age < 45:
        return '30-44'
    elif age < 60:
        return '45-59'
    else:
        return '60+'
# Function to recode citizenship
def recode_citizenship(cit):
    if cit == 'Denmark':
        return 'Denmark'
    else:
        return 'Other'

# Function to recode marital status
def recode_marital(status):
    if status == 'Widowed' or status == 'Divorced':
        return 'Divorced/Widowed'
    else:
        return status

# Function to convert last_voted to evote
def convert_evote(last_voted):
    if last_voted == 0:
        return 0  # paper
    elif last_voted == 1:
        return 1  # electronic
    else:
        return None  # didn't vote


print(register['last_voted'].value_counts())
print(anon_data['evote'].value_counts())

def find_matches(person_row, anon_df):
    """
    person row: row from register with transformed QIs
    anon_df: anonymised dataset
    Returns: list of matching row indices
    """
    matches = []
    for idx, anon_row in anon_df.iterrows():
        match = True

        #check each quasi-identifier
        #sex - direct match
        if person_row['sex'] != anon_row['sex']:
            match = False
            continue
            
        # Age band - direct match
        if person_row['age_band'] != anon_row['age']:
            match = False
            continue

        # ZIP - direct match
        if person_row['zip'] != anon_row['zip']:
            match = False
            continue

        # Voting channel - direct match
        if person_row['evote'] != anon_row['evote']:
            match = False
            continue

        # Citizenship - blank matches anything
        if pd.notna(anon_row['citizenship']) and anon_row['citizenship'] != '':
            if person_row['citizenship'] != anon_row['citizenship']:
                match = False
                continue

        # Marital status - blank matches anything
        if pd.notna(anon_row['marital_status']) and anon_row['marital_status'] != '':
            if person_row['marital_status'] != anon_row['marital_status']:
                match = False
                continue    
        
        if match:
            matches.append(idx)
    
    return matches


last_voted
0    779
2    476
1    334
Name: count, dtype: int64
evote
0    147
1     53
Name: count, dtype: int64


## Select respondents from the register and recode them

In [ ]:
# Keep only people who voted (last_voted: 0 = paper, 1 = e-vote, 2 = did not vote)
voters = register[register['last_voted'] != 2].copy()

# Transform register rows into the released format
voters['evote'] = voters['last_voted'].apply(convert_evote)
voters['age_band'] = voters['dob'].apply(dob_to_age_band)
voters['citizenship'] = voters['citizenship'].apply(recode_citizenship)
voters['marital_status'] = voters['marital_status'].apply(recode_marital)

# Keep only people on the leaked survey list
people = voters[voters['name'].isin(survey_names)].reset_index(drop=True)

print("Voters:", len(voters))
print("Rows for leaked names:", len(people))

## Check our recoding against Group N's risk table

In [ ]:
QI = ['sex', 'zip', 'evote', 'age_band', 'citizenship', 'marital_status']

# k = number of voters sharing a person's six recoded values
k = voters.groupby(QI).size().rename('k')
people = people.join(k, on=QI)

print("Respondents with k = 1:", (people['k'] == 1).sum())
print("Respondents with k < 3:", (people['k'] < 3).sum())

## Find candidate records for each person

In [ ]:
candidates = {}
for idx, person_row in people.iterrows():
    candidates[idx] = set(find_matches(person_row, anon_data))

print("People with one candidate:", sum(len(c) == 1 for c in candidates.values()))
print("People with no candidate:", sum(len(c) == 0 for c in candidates.values()))

## Remove records that must belong to someone else

In [ ]:
def prune(candidates):
    """
    candidates: dict of person -> set of candidate record indices
    Returns: new dict where a record that is the only candidate of one person
             is removed from everyone else, repeated until nothing changes
    """
    pruned = {person: set(recs) for person, recs in candidates.items()}
    changed = True
    while changed:
        changed = False
        for person, recs in pruned.items():
            if len(recs) == 1:
                record = next(iter(recs))
                for other, other_recs in pruned.items():
                    if other != person and record in other_recs:
                        other_recs.remove(record)
                        changed = True
    return pruned

feasible = prune(candidates)

print("Candidate sets shrunk:", sum(feasible[p] != candidates[p] for p in candidates))
print("People with one candidate:", sum(len(c) == 1 for c in feasible.values()))

# Candidate-set sizes before and after pruning (respondents only)
respondents = [p for p in feasible if feasible[p]]
sizes = pd.DataFrame({
    'before': [len(candidates[p]) for p in respondents],
    'after': [len(feasible[p]) for p in respondents],
})
print(sizes.apply(pd.Series.value_counts).fillna(0).astype(int).T)

## Read off each person's vote and save the results

In [ ]:
results = []
for idx, person_row in people.iterrows():
    recs = sorted(feasible[idx])
    parties = set(anon_data.loc[recs, 'party'])

    # Skip people with no candidates (not a respondent) or with both parties possible
    if len(parties) != 1:
        continue

    if len(recs) == 1:
        status = 'certain (single record)'
        education = anon_data.loc[recs[0], 'education']
    else:
        status = 'certain (homogeneous group)'
        education = ''

    results.append({
        'name': person_row['name'],
        'dob': person_row['dob'].date(),
        'zip': person_row['zip'],
        'party': parties.pop(),
        'status': status,
        'n_candidate_records': len(recs),
        'education': education,
    })

results = pd.DataFrame(results).sort_values(['status', 'name'])
results.to_csv('submission_results.csv', index=False)

print(results['status'].value_counts())
print(results['party'].value_counts())
print("Not determined:", len(respondents) - len(results))

## Count single-party groups in the released data

In [ ]:
# Group released records by their visible quasi-identifiers (blank counted as its own value)
anon_QI = ['sex', 'zip', 'evote', 'age', 'citizenship', 'marital_status']
groups = anon_data.fillna('*').groupby(anon_QI)['party'].agg(['size', 'nunique'])
single_party = groups[groups['nunique'] == 1]

print("Groups:", len(groups))
print("Single-party groups:", len(single_party))
print("Records in single-party groups:", single_party['size'].sum())